# Semana 41 · ¿Se está expandiendo el universo?

Este cuaderno acompaña a la sesión de proyecto de la semana 41: medir el corrimiento al rojo de un espectro, construir un diagrama de Hubble con errores y estimar la constante de Hubble y la edad del universo. **Los datos son simulados**, con valores realistas, para que puedas comparar el resultado con la respuesta verdadera.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Medir el corrimiento al rojo

Un espectro **sintético** de galaxia con líneas de emisión de hidrógeno (H-alfa, H-beta) y de oxígeno ionizado dos veces ([O III], dos líneas en 495,9 y 500,7 nm, muy típicas de las galaxias que forman estrellas). Lo desplazamos con un $z$ que el cuaderno «esconde» y lo medimos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(41)
lineas = {"H-beta": 486.13, "[O III] 495,9": 495.89, "[O III] 500,7": 500.69, "H-alfa": 656.28}
intensidad = {"H-beta": 0.35, "[O III] 495,9": 0.3, "[O III] 500,7": 0.9, "H-alfa": 1.0}

z_escondido = rng.uniform(0.02, 0.08)
lam = np.arange(400, 800, 0.1)
flujo = 0.2 + 0.0002 * (lam - 400)
for nombre, l0 in lineas.items():
    flujo += intensidad[nombre] * np.exp(-0.5 * ((lam - l0 * (1 + z_escondido)) / 0.4) ** 2)
flujo += rng.normal(0, 0.02, lam.size)

plt.figure(figsize=(12, 4))
plt.plot(lam, flujo, lw=0.7)
for l0 in lineas.values():
    plt.axvline(l0, color="gray", ls=":", lw=0.8)
plt.xlabel("longitud de onda observada (nm)"); plt.ylabel("flujo")
plt.title("Espectro sintético de una galaxia (líneas punteadas: posiciones en reposo)"); plt.show()

Las cuatro líneas aparecen desplazadas a la derecha de sus posiciones en reposo, todas en el mismo factor $1 + z$. Para no confundir unas líneas con otras, primero buscamos el $z$ que mejor encaja con **todas a la vez**: probamos muchos valores de $z$ y, para cada uno, sumamos el flujo en las posiciones donde caerían las cuatro líneas. El $z$ con la suma mayor es la primera estimación. (Es una versión sencilla de la **correlación cruzada** con una plantilla, el método que usan los grandes cartografiados de galaxias.) Después afinamos buscando el pico de cada línea cerca de su posición prevista.

In [ ]:
reposo = np.array(list(lineas.values()))
z_prueba = np.arange(0, 0.15, 0.0001)
puntuacion = [np.interp(reposo * (1 + z), lam, flujo).sum() for z in z_prueba]
z0 = z_prueba[np.argmax(puntuacion)]

plt.figure(figsize=(8, 3.5))
plt.plot(z_prueba, puntuacion); plt.axvline(z0, color="r", ls="--")
plt.xlabel("z de prueba"); plt.ylabel("flujo sumado en las 4 posiciones"); plt.grid(alpha=0.3); plt.show()
print(f"primera estimación: z = {z0:.4f}")

zs = []
for nombre, l0 in lineas.items():
    prevista = l0 * (1 + z0)
    sel = np.abs(lam - prevista) < 2
    l_obs = lam[sel][np.argmax(flujo[sel])]
    z = (l_obs - l0) / l0
    zs.append(z)
    print(f"{nombre:<15} reposo {l0:7.2f} nm   observada {l_obs:7.2f} nm   z = {z:.5f}")
zs = np.array(zs)
print(f"\nz medio = {zs.mean():.5f} ± {zs.std(ddof=1) / np.sqrt(len(zs)):.5f}   (escondido: {z_escondido:.5f})")
print(f"velocidad ≈ c·z = {299792 * zs.mean():.0f} km/s")

## 2. Un diagrama de Hubble simulado

Simulamos 40 galaxias con distancias entre 10 y 400 Mpc. Cada una se aleja con la velocidad de la expansión, $H_0\,d$ con un $H_0$ verdadero de 70 km/s/Mpc, más una **velocidad peculiar** aleatoria de unos 300 km/s (su movimiento propio dentro de su grupo o cúmulo). Las distancias tienen un error del 8 % (como con supernovas de tipo Ia) y las velocidades, de 50 km/s.

In [ ]:
H0_real = 70.0
n = 40
d_real = rng.uniform(10, 400, n)                     # Mpc
v = H0_real * d_real + rng.normal(0, 300, n) + rng.normal(0, 50, n)
sigma_d = 0.08 * d_real
d_med = d_real + rng.normal(0, 1, n) * sigma_d

plt.figure(figsize=(9, 6))
plt.errorbar(d_med, v, xerr=sigma_d, fmt="o", ms=4, capsize=2, alpha=0.8)
plt.xlabel("distancia (Mpc)"); plt.ylabel("velocidad de alejamiento (km/s)")
plt.title("Diagrama de Hubble (datos simulados)"); plt.grid(alpha=0.3); plt.show()

## 3. Ajustar $H_0$

La ley es una recta que pasa por el origen, $v = H_0 d$. Ajustamos de tres maneras: sin pesos, con pesos y con la recta general $v = a + b\,d$ para comprobar que $a$ sale compatible con cero. Para el ajuste ponderado, cada galaxia tiene un error total en velocidad que combina el de su velocidad (peculiar y de medida) y el de su distancia traducido a velocidad, $H_0\,\sigma_d$.

In [ ]:
# 1) Recta por el origen sin pesos: minimizar Σ(v - H d)² da H = Σ v d / Σ d²
H_simple = (v * d_med).sum() / (d_med**2).sum()

# 2) Ponderada: minimizar Σ((v - H d)/σ)² da H = Σ(v d/σ²) / Σ(d²/σ²)
sigma_v = np.sqrt(300**2 + 50**2 + (H_simple * sigma_d) ** 2)
w = 1 / sigma_v**2
H_pond = (w * v * d_med).sum() / (w * d_med**2).sum()
sigma_H = 1 / np.sqrt((w * d_med**2).sum())
chi2 = (((v - H_pond * d_med) / sigma_v) ** 2).sum()

# 3) Recta general con polyfit (pesos 1/σ)
(b, a), cov = np.polyfit(d_med, v, 1, w=1 / sigma_v, cov="unscaled")

print(f"sin pesos:           H0 = {H_simple:.1f} km/s/Mpc")
print(f"ponderado:           H0 = {H_pond:.1f} ± {sigma_H:.1f} km/s/Mpc   (verdadero: {H0_real})")
print(f"recta general:       b = {b:.1f} ± {np.sqrt(cov[0, 0]):.1f},  a = {a:.0f} ± {np.sqrt(cov[1, 1]):.0f} km/s")
print(f"chi² = {chi2:.1f} con {n - 1} grados de libertad")

El $\chi^2$ sale cerca del número de puntos menos uno (hemos ajustado un parámetro): los errores que hemos supuesto describen bien la dispersión. La ordenada $a$ es compatible con cero dentro de su error, como debe ser si la ley pasa por el origen.

## 4. La edad del universo

In [ ]:
km_por_Mpc = 3.0857e19
segundos_por_ano = 3.156e7
def tiempo_hubble(H0):
    return km_por_Mpc / H0 / segundos_por_ano / 1e9    # miles de millones de años

t = tiempo_hubble(H_pond)
dt = t * sigma_H / H_pond                               # propagación de errores: error relativo igual
print(f"1/H0 = {t:.1f} ± {dt:.1f} miles de millones de años")
for H in [67.4, 70, 73.0]:
    print(f"  con H0 = {H}: {tiempo_hubble(H):.1f} miles de millones de años")

## 5. La tensión de Hubble

Dos medidas publicadas: la del equipo SH0ES con la escalera de distancias (Riess y colaboradores, 2022) y la de la misión Planck con el fondo cósmico (2018). Las dibujamos con sus barras de error de 1σ.

In [ ]:
medidas = [("Planck 2018 (fondo cósmico)", 67.4, 0.5), ("SH0ES 2022 (escalera de distancias)", 73.04, 1.04),
           ("este cuaderno (simulado)", H_pond, sigma_H)]
fig, ax = plt.subplots(figsize=(8, 3))
for i, (nombre, H, s) in enumerate(medidas):
    ax.errorbar(H, i, xerr=s, fmt="o", capsize=4)
    ax.text(H, i + 0.2, nombre, ha="center", fontsize=8)
ax.set_yticks([]); ax.set_ylim(-0.5, len(medidas) - 0.2)
ax.set_xlabel("H0 (km/s/Mpc)"); ax.grid(alpha=0.3, axis="x"); plt.tight_layout(); plt.show()

diferencia = 73.04 - 67.4
sigma_dif = np.sqrt(1.04**2 + 0.5**2)
print(f"diferencia: {diferencia:.2f} ± {sigma_dif:.2f} → {diferencia / sigma_dif:.1f} desviaciones típicas")

## 6. Experimenta

1. Simula solo galaxias cercanas, entre 1 y 20 Mpc. ¿Qué le pasa a la precisión de $H_0$? ¿Por qué las velocidades peculiares estropean la medida a distancias pequeñas?
2. Introduce un error sistemático: multiplica todas las distancias por 1,1 (una calibración un 10 % equivocada). ¿Qué $H_0$ sale? ¿Lo detecta el $\chi^2$?
3. En el espectro, aumenta el ruido a 0,2. ¿Qué línea deja de medirse bien primero?

In [ ]:
# Tu experimento aquí